# Fine-Tuning Laya for GitHub Issue Triage (Kaggle 2×T4 DDP)

Adapted from the upstream notebook
[`laya_finetune_typed_decisions_2xT4_kaggle.ipynb`](https://github.com/NandhaKishorM/laya/blob/main/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb).
The DDP training script is unchanged; only the dataset cells differ.

**Task.** Two typed questions about a freshly opened GitHub issue:

| question | type | answers |
|---|---|---|
| `issue_type` | `choice` | `bug`, `feature`, `question`, `docs` |
| `needs_more_info` | `noul` | `true` / `false` |

**Data.** ~7.7k training issues hand-labeled by maintainers across 14 public
repos, balanced across the four types. Three repos (`huggingface/transformers`,
`facebook/react`, `microsoft/TypeScript`) are held out entirely, so the test
number measures generalization to an unseen project.

### ⚠️ Kaggle settings

`/kaggle/working` does **not** survive powering the session off -- a finished
model is lost with it. Section 5b pushes the checkpoint to the Hub as soon as
training ends; do not skip it.

* **Accelerator:** `GPU T4 x2`
* **Internet:** `On`


## 1. Environment & Dual T4 GPU Check
Verify both T4 GPUs are detected.


In [ ]:
!nvidia-smi
import os, torch

# Upstream targets Kaggle. Creating this directory on other hosts lets every
# hardcoded /kaggle/working path -- including the one inside train_ddp.py --
# work unchanged on Colab.
os.makedirs("/kaggle/working", exist_ok=True)

N_GPU = torch.cuda.device_count()
print(f"CUDA available: {torch.cuda.is_available()} | visible GPUs: {N_GPU}")
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert N_GPU >= 1, (
    "No GPU detected.\n"
    "  Kaggle: right sidebar -> Accelerator -> GPU T4 x2 (preferred)\n"
    "  Colab:  Runtime -> Change runtime type -> T4 GPU"
)
if N_GPU == 1:
    print("\nOne GPU detected: training runs single-process and takes roughly twice "
          "the 4-5 hours upstream quotes for 2xT4. Kaggle's GPU T4 x2 is the faster path.")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


## 2. Install Dependencies


In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy accelerate tabulate
import laya, transformers, datasets, torch
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)


## 3. Load & Preprocess the Issue-Triage Data
Set `DATA_BASE` to your repo. Gold is per-question, so issues that answer only one of the two questions still contribute that one.


In [ ]:
import os, json, torch
from datasets import load_dataset
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

MODEL_ID = "convaiinnovations/laya"
# Point this at your fork if you changed the dataset.
DATA_BASE = "https://raw.githubusercontent.com/manyamkarthik/laya-issue-triage/main/data/processed"

print(f"Fetching tokenizer and config from {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

print("Loading the issue-triage splits...")
ds_train = load_dataset("json", data_files=f"{DATA_BASE}/train.jsonl.gz", split="train")
print(ds_train)


def build_training_item(state, q, gold_q):
    """Unchanged from upstream, except that a question absent from gold is
    skipped by the caller rather than assumed present."""
    t = q["type"]
    crit = q.get("criteria", {})
    if t == "choice":
        keys = list(crit.keys())
        target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    elif t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    elif t == "score":
        n_levels = len(crit) if isinstance(crit, list) else 4
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(n_levels)]

    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))

    seq, markers = build_sequence(tok, state, {"t": t, "ins": q["instructions"], "crit": crit}, cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {"ids": seq, "markers": markers, "qtype": QTYPES[t], "target": target, "label": label}


items, skipped = [], 0
for row in ds_train:
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    for qid, q in questions.items():
        # Rows carry gold only for the questions their labels actually answer;
        # `needs_more_info` is absent for projects that do not use such labels.
        if qid not in gold:
            skipped += 1
            continue
        it = build_training_item(state, q, gold[qid])
        if it:
            items.append(it)
        else:
            skipped += 1

print(f"Preprocessed {len(items)} training sequences from {len(ds_train)} issues "
      f"({skipped} questions skipped for missing gold).")
torch.save(items, "/kaggle/working/train_items.pt")
print("Saved to /kaggle/working/train_items.pt")


## 4. DDP Training Script (`train_ddp.py`)
We write the multi-GPU distributed RLCD training script using pure policy gradients with proper scoring rules and DDP gradient synchronization.


In [ ]:
%%writefile /kaggle/working/train_ddp.py
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward, QTYPES

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items])
    }

def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    
    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)
    
    all_items = torch.load("/kaggle/working/train_items.pt", weights_only=False)
    my_items = all_items[rank::world_size]
    
    # Our first run peaked at epoch 2, then loss rose and the policy went
    # confidently wrong on hard batches. Override with LAYA_EPOCHS.
    EPOCHS = int(os.environ.get("LAYA_EPOCHS", "2"))
    MICRO_BATCH = 8      # 8 sequences per forward pass per GPU
    GRAD_ACCUM = 4       # Effective batch across 2 GPUs = 64 sequences (8 * 2 * 4)
    GROUP_SIZE = 4       # GRPO baseline samples
    LR_ENCODER = 2.5e-5  # Encoder adaptation rate
    LR_HEAD = 1.0e-4     # Head adaptation rate
    SIGMA_START = 0.4    # Exploration noise
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)
    
    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    
    if rank == 0:
        print(f"Starting 2xT4 DDP training: {len(all_items)} total items | {len(my_items)} per rank | {EPOCHS} epochs")
    t0 = time.time()
    
    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0
        
        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress
        
        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue
            
            batch = collate_train_batch(chunk, tok.pad_token_id)
            
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )
            
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)
            
            # 1. Sample G noisy logit distributions with zero-mean projection
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            
            # 2. Evaluate proper scoring reward (w_sph=0.75 for soft target matching)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)
            
            # 3. Policy gradient loss + full 1.0 soft cross-entropy guidance
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()
            
            scaler.scale(loss).backward()
            accum_step += 1
            
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            
            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            
            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item()*GRAD_ACCUM:.4f} | Reward: {r.mean().item():.3f} | LR: {cur_lr:.2e}")

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()

        # Overwrite a single rolling checkpoint after each epoch so a crash,
        # OOM, or Kaggle session timeout doesn't lose all prior training.
        if rank == 0:
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            os.makedirs(ckpt_dir, exist_ok=True)
            ckpt_sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
            save_file(ckpt_sd, os.path.join(ckpt_dir, "model.safetensors"))
            model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
            tok.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))
            with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
                json.dump({
                    "epoch": epoch + 1,
                    "total_epochs": EPOCHS,
                    "avg_loss": epoch_loss / max(1, n_batches)
                }, f, indent=2)
            print(f"  Saved rolling checkpoint (epoch {epoch+1}/{EPOCHS}) to {ckpt_dir}")

    dist.barrier()
    
    # Post-training temperature calibration on rank 0 (micro-batched in chunks of 16 to prevent OOM)
    if rank == 0:
        print("\nFitting post-training calibration temperatures...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        calib_items = all_items[::15][:400]
        calib_preds = []
        with torch.no_grad():
            for c_idx in range(0, len(calib_items), 16):
                c_chunk = calib_items[c_idx:c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)
                with torch.autocast("cuda", dtype=torch.float16):
                    l_sub, _ = model(
                        cb["input_ids"].to(device),
                        cb["attention_mask"].to(device),
                        cb["marker_pos"].to(device),
                        cb["marker_mask"].to(device),
                        cb["qtype"].to(device)
                    )
                l_np = l_sub.float().cpu().numpy()
                for r, it in enumerate(c_chunk):
                    k = len(it["markers"])
                    calib_preds.append((it["qtype"], l_np[r, :k], it["target"]))
        
        fitted_temps = [1.2, 1.2, 1.2]
        try:
            for qt in range(3):
                sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
                if sel:
                    fitted_temps[qt] = fit_one_temp(sel)
            print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
        os.makedirs(output_dir, exist_ok=True)
        sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
        save_file(sd, os.path.join(output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
        
        cfg["fine_tuned"] = True
        cfg["model_name"] = "laya-typed-decisions"
        cfg["temperature"] = fitted_temps
        with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Model successfully saved to {output_dir}!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()


## 5. Launch Multi-GPU Fine-Tuning with `torchrun`
Runs on both T4 GPUs in parallel (~4 to 6 minutes total).


In [ ]:
OUTPUT_DIR = "/kaggle/working/laya_issue_triage"
MODEL_DIR = model_dir

cmd = (f"torchrun --standalone --nproc_per_node={N_GPU} "
       f"/kaggle/working/train_ddp.py {MODEL_DIR} {OUTPUT_DIR}")
print(f"Launching training on {N_GPU} GPU(s):", cmd)
!{cmd}


## 5b. Save the Checkpoint Off-Box *Immediately*

**Run this before anything else.** Kaggle keeps `/kaggle/working` only for the
life of the session: power the notebook off, or let it time out, and a finished
model is gone with it. Only a "Save Version" run persists outputs.

This pushes the raw weights to a **private** Hub repo the moment training ends,
so evaluation and publishing can happen later, in any session, without retraining.


In [ ]:
import os
from huggingface_hub import HfApi, create_repo

HF_USER = "harikarthikmanyam"        # <- your HF handle
HF_MODEL = f"{HF_USER}/laya-issue-triage"


def _hf_token():
    """Kaggle secrets, then Colab secrets, then an env var."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        raise RuntimeError(
            "No HF_TOKEN found. Kaggle: Add-ons -> Secrets. "
            "Colab: the key icon in the left sidebar."
        )
    return tok


HF_TOKEN = _hf_token()

assert os.path.isdir(OUTPUT_DIR), f"{OUTPUT_DIR} is missing -- training did not finish."

# Private for now: the model card with real numbers goes up in section 8, once
# the evaluation has actually been read.
create_repo(HF_MODEL, token=HF_TOKEN, private=True, exist_ok=True)
HfApi().upload_folder(folder_path=OUTPUT_DIR, repo_id=HF_MODEL, token=HF_TOKEN,
                      commit_message="raw checkpoint, pre-evaluation")
print(f"Checkpoint safe at https://huggingface.co/{HF_MODEL} (private)")
print("A later session can now evaluate with: laya.Agent(HF_MODEL, token=HF_TOKEN)")


## 6. Evaluate on the Held-Out Repos
These three repos appear nowhere in training, so this is a generalization number, not a memorization one.


In [ ]:
import time, json
import numpy as np
import pandas as pd
from datasets import load_dataset
import laya

print("Loading held-out test split (3 repos never seen in training)...")
ds_test = load_dataset("json", data_files=f"{DATA_BASE}/test.jsonl.gz", split="train")

agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")

predictions = []
latencies_ms = []

print(f"Evaluating {len(ds_test)} test cases on GPU...")
t0_eval = time.time()

for i, row in enumerate(ds_test):
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])

    t0 = time.perf_counter()
    res = agent_ft.predict(state, questions)
    dt_ms = (time.perf_counter() - t0) * 1000
    latencies_ms.append(dt_ms)

    predictions.append({
        "id": row["id"],
        "workflow": row["workflow"],   # the repo, for a per-repo breakdown
        "pred": res["answers"],
        "gold": gold,
        # Score only the questions this issue actually answers.
        "questions": {k: v for k, v in questions.items() if k in gold},
        "latency_ms": dt_ms,
    })

print(f"Evaluated all {len(ds_test)} cases in {time.time() - t0_eval:.1f}s")


## 7. Metrics Against Baselines For This Task

Upstream's notebook compares against TypeSafe Jev on the `typed-decisions`
benchmark. Those numbers are **not** comparable to this one — different task,
different label space, different data — so they are replaced here with
baselines measured on this exact test set:

* **random** — uniform over the four types
* **majority class** — always answer the most common label
* **base `laya`, zero-shot** — the released checkpoint, no fine-tuning

The last one is the honest "before" column, measured with
`scripts/eval_laya.py --model convaiinnovations/laya`.


In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from laya.common import ece_score

TYPES = ["bug", "feature", "question", "docs"]


def macro_f1(pairs, labels):
    out = []
    for lb in labels:
        tp = sum(1 for g, p in pairs if g == lb and p == lb)
        fp = sum(1 for g, p in pairs if g != lb and p == lb)
        fn = sum(1 for g, p in pairs if g == lb and p != lb)
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        out.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    return float(np.mean(out)), dict(zip(labels, [round(v, 3) for v in out]))


type_pairs, info_pairs = [], []
type_confs, type_correct, info_confs, info_correct = [], [], [], []
per_repo = {}

for item in predictions:
    pred, gold, repo = item["pred"], item["gold"], item["workflow"]
    if "issue_type" in gold:
        p = pred["issue_type"]["choice"]
        g = gold["issue_type"]["label"]
        type_pairs.append((g, p))
        type_confs.append(max(pred["issue_type"]["probabilities"].values()))
        type_correct.append(float(p == g))
        hit, n = per_repo.get(repo, (0, 0))
        per_repo[repo] = (hit + float(p == g), n + 1)
    if "needs_more_info" in gold:
        p_true = pred["needs_more_info"]["noul"]
        p = "true" if p_true >= 0.5 else "false"
        g = gold["needs_more_info"]["label"]
        info_pairs.append((g, p))
        info_confs.append(max(p_true, 1 - p_true))
        info_correct.append(float(p == g))

type_acc = float(np.mean(type_correct))
info_acc = float(np.mean(info_correct))
type_f1, per_class = macro_f1(type_pairs, TYPES)
info_f1, _ = macro_f1(info_pairs, ["true", "false"])

type_majority = Counter(g for g, _ in type_pairs).most_common(1)[0][1] / len(type_pairs)
info_majority = max(
    sum(1 for g, _ in info_pairs if g == lb) / len(info_pairs) for lb in ("true", "false")
)

# Measured locally on this test set before fine-tuning; re-run
# scripts/eval_laya.py if the dataset changes.
BASE_ZEROSHOT = {"issue_type_acc": 0.626, "issue_type_f1": 0.524, "needs_info_acc": 0.563}

table = [
    {"Model": "random", "issue_type acc": round(1 / 4, 3), "issue_type macro-F1": "-",
     "needs_info acc": 0.5, "ms/case": "-"},
    {"Model": "majority class", "issue_type acc": round(type_majority, 3),
     "issue_type macro-F1": "-", "needs_info acc": round(info_majority, 3), "ms/case": "-"},
    {"Model": "base laya (zero-shot)", "issue_type acc": BASE_ZEROSHOT["issue_type_acc"],
     "issue_type macro-F1": BASE_ZEROSHOT["issue_type_f1"],
     "needs_info acc": BASE_ZEROSHOT["needs_info_acc"], "ms/case": "-"},
    {"Model": "laya fine-tuned (this run)", "issue_type acc": round(type_acc, 3),
     "issue_type macro-F1": round(type_f1, 3), "needs_info acc": round(info_acc, 3),
     "ms/case": round(float(np.percentile(latencies_ms, 50)), 1)},
]

print("=== ISSUE TRIAGE, HELD-OUT REPOS ===\n")
print(pd.DataFrame(table).to_markdown(index=False))

print("\nper-class F1 (issue_type):", per_class)
print("needs_more_info macro-F1:", round(info_f1, 3))
print("ECE issue_type:", round(float(ece_score(np.array(type_confs), np.array(type_correct))), 3))
print("ECE needs_more_info:", round(float(ece_score(np.array(info_confs), np.array(info_correct))), 3))

print("\nper-repo issue_type accuracy (never seen in training):")
for repo, (hit, n) in sorted(per_repo.items()):
    print(f"  {repo:<32} {hit / n:.3f}  (n={n})")

print("\nconfusion (gold -> pred):")
conf = Counter(type_pairs)
for g in TYPES:
    print(f"  {g:>8}  " + "  ".join(f"{p}:{conf.get((g, p), 0):4d}" for p in TYPES))


## 8. Push to the Hub

The model card is what search engines and AI assistants actually read, so it is
generated from the numbers this run measured -- with the task scope stated
explicitly, and no comparison to benchmarks this model was never evaluated on.


In [ ]:
import os, json
from huggingface_hub import HfApi, create_repo

HF_USER = "harikarthikmanyam"        # <- your HF handle
HF_MODEL = f"{HF_USER}/laya-issue-triage"
GITHUB_REPO = "manyamkarthik/laya-issue-triage"

# Token lookup: Kaggle secrets, then Colab secrets, then an env var.
def _hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        raise RuntimeError(
            "No HF_TOKEN found. Kaggle: Add-ons -> Secrets. "
            "Colab: the key icon in the left sidebar."
        )
    return tok

HF_TOKEN = _hf_token()

per_repo_rows = "\n".join(
    f"| `{repo}` | {hit / n:.3f} | {n} |" for repo, (hit, n) in sorted(per_repo.items())
)
per_class_rows = "\n".join(f"| `{k}` | {v:.3f} |" for k, v in per_class.items())

card = f"""---
license: apache-2.0
base_model: convaiinnovations/laya
library_name: laya
pipeline_tag: text-classification
tags:
  - laya
  - typed-decisions
  - github
  - issue-triage
  - non-autoregressive
  - modernbert
metrics:
  - accuracy
  - f1
model-index:
  - name: laya-issue-triage
    results:
      - task:
          type: text-classification
          name: GitHub issue type classification
        dataset:
          name: laya-issue-triage (held-out repos)
          type: {GITHUB_REPO}
        metrics:
          - type: accuracy
            value: {type_acc:.4f}
          - type: f1
            name: macro F1
            value: {type_f1:.4f}
---

# laya-issue-triage

[Laya](https://github.com/NandhaKishorM/laya) fine-tuned to triage GitHub
issues. It answers two typed questions about an issue in a single forward pass,
with no text generation:

| question | type | answers |
|---|---|---|
| `issue_type` | `choice` | `bug`, `feature`, `question`, `docs` |
| `needs_more_info` | `noul` | `true` / `false` |

## Results

Measured on {len(ds_test)} issues from three repositories held out of training
entirely (`huggingface/transformers`, `facebook/react`, `microsoft/TypeScript`),
so these are generalization numbers, not memorization.

| model | issue_type acc | issue_type macro-F1 | needs_info acc |
|---|---|---|---|
| random | 0.250 | - | 0.500 |
| majority class | {type_majority:.3f} | - | {info_majority:.3f} |
| base `laya`, zero-shot | {BASE_ZEROSHOT["issue_type_acc"]:.3f} | {BASE_ZEROSHOT["issue_type_f1"]:.3f} | {BASE_ZEROSHOT["needs_info_acc"]:.3f} |
| **this model** | **{type_acc:.3f}** | **{type_f1:.3f}** | **{info_acc:.3f}** |

Per-class F1 on `issue_type`:

| class | F1 |
|---|---|
{per_class_rows}

Per-repository accuracy on `issue_type`:

| repository | accuracy | n |
|---|---|---|
{per_repo_rows}

Median latency on a T4 during evaluation: {float(np.percentile(latencies_ms, 50)):.1f} ms per issue.

### Scope of these numbers

This model is evaluated **only** on GitHub issue triage. It has not been run on
the `LocalLLaMA/typed-decisions` benchmark, so its scores are not comparable to
numbers reported there for the base Laya checkpoints or for any other system.
Different task, different label space, different data.

## Usage

```python
import laya

agent = laya.Agent("{HF_MODEL}")
questions = {json.dumps(json.loads(ds_test[0]["questions"]), indent=4)}

state = {{"title": "Crash when opening a file with a BOM",
         "body": "v1.4.2 on Linux. Steps: open any UTF-8-BOM file, editor segfaults."}}
result = agent.predict(state, questions)
print(result["answers"]["issue_type"]["choice"])
```

The question definitions above are part of the contract: this model was trained
against those exact `instructions` and `criteria` strings, so changing them
changes behaviour.

## Training data

~7.7k closed issues carrying maintainer-applied labels, harvested from 14
public repositories and balanced across the four types. Labels come from the
maintainers who triaged each issue, not from a teacher model.

Dataset, harvesting code and evaluation script:
https://github.com/{GITHUB_REPO}

## Limitations

- English issue text only; use `laya-multilingual` as a base for other languages.
- Issue text is truncated to roughly 900 characters, so decisions rest on the
  title and opening paragraphs.
- `needs_more_info` is supervised by maintainer labels such as `needs-repro`,
  which projects apply inconsistently; it is a weaker signal than `issue_type`.
- Confidences are uncalibrated unless you fit temperatures (see the notebook).
"""

create_repo(HF_MODEL, token=HF_TOKEN, exist_ok=True)
api = HfApi()
# Section 5b created this private. Publishing is the deliberate step, taken only
# after the numbers above have been read.
api.update_repo_settings(repo_id=HF_MODEL, private=False, token=HF_TOKEN)
api.upload_folder(folder_path=OUTPUT_DIR, repo_id=HF_MODEL, token=HF_TOKEN)
Path_card = "/kaggle/working/README_card.md"
with open(Path_card, "w") as f:
    f.write(card)
api.upload_file(path_or_fileobj=Path_card, path_in_repo="README.md",
                repo_id=HF_MODEL, token=HF_TOKEN,
                commit_message=f"issue_type acc {type_acc:.3f}, macro-F1 {type_f1:.3f}")
print(f"pushed https://huggingface.co/{HF_MODEL}")


## 9. Save the Benchmark Report
Commit this JSON back to the repo so the README table has a machine-readable source.


In [ ]:
import json
import numpy as np

report = {
    "task": "github-issue-triage",
    "base_model": "convaiinnovations/laya",
    "n_test_issues": len(ds_test),
    "held_out_repos": sorted(per_repo.keys()),
    "issue_type": {
        "n": len(type_pairs),
        "accuracy": round(type_acc, 4),
        "macro_f1": round(type_f1, 4),
        "per_class_f1": per_class,
        "majority_baseline": round(type_majority, 4),
        "random_baseline": 0.25,
        "base_zeroshot": BASE_ZEROSHOT["issue_type_acc"],
    },
    "needs_more_info": {
        "n": len(info_pairs),
        "accuracy": round(info_acc, 4),
        "macro_f1": round(info_f1, 4),
        "majority_baseline": round(info_majority, 4),
        "base_zeroshot": BASE_ZEROSHOT["needs_info_acc"],
    },
    "latency_ms": {
        "p50": round(float(np.percentile(latencies_ms, 50)), 1),
        "p95": round(float(np.percentile(latencies_ms, 95)), 1),
        "device": "T4",
    },
    "per_repo_issue_type_accuracy": {r: round(h / n, 4) for r, (h, n) in sorted(per_repo.items())},
    "scope_note": (
        "Evaluated only on GitHub issue triage. Not run on LocalLLaMA/typed-decisions; "
        "not comparable to numbers reported on that benchmark."
    ),
}

with open("/kaggle/working/benchmark_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))
